In [1]:
import openai, json, requests

client = openai.OpenAI()
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [2]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

SYSTEM_PROMPT = """You are a Movie Expert Agent.
You have access to the following functions:
- get_popular_movies(): Returns a list of currently popular movies.
- get_movie_details(id): Returns detailed information about a specific movie by its ID.
- get_movie_credits(id): Returns the cast and crew of a specific movie by its ID.
Use the appropriate function to answer the user's question about movies."""

In [4]:
def process_ai_response(message, messages):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Calling function: {function_name} with args: {arguments}")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        call_ai(messages)
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai(messages):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message, messages)

In [5]:
# Test 1: 인기 영화 목록
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
user_input = "지금 인기 있는 영화가 무엇인지 알려줘"
print(f"User: {user_input}")
messages.append({"role": "user", "content": user_input})
call_ai(messages)

User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with args: {}
AI: 현재 인기 있는 영화는 다음과 같습니다:

1. **Mercy**
   - 개요: 가까운 미래, 한 탐정이 아내 살해 혐의로 재판을 받고 있으며, 자신을 지지했던 고급 AI 판사에게 90분 안에 자신의 무죄를 증명해야 합니다.
   - 개봉일: 2026-01-20
   - 평점: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple**
   - 개요: Dr. Kelson은 세상을 바꿀 수 있는 충격적인 새로운 관계를 발견하게 되고, Spike는 Jimmy Crystal을 만나는 꿈에서 날카로운 악몽을 경험합니다.
   - 개봉일: 2026-01-14
   - 평점: 7.2
   - ![포스터](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Les Orphelins**
   - 개요: 두 유년 친구가 다시 만나면서, 상대방과 정반대의 삶을 살게 되고, 그들의 첫사랑이 의심스러운 사고로 사망하게 됩니다.
   - 개봉일: 2025-08-20
   - 평점: 6.0
   - ![포스터](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **A Woman Scorned**
   - 개요: 가족 주말에 지역 남자들로부터 위협을 받고, 자매가 공격받고 살해됨으로써 복수를 결심한 여성이 그 남자들을 추적하는 이야기입니다.
   - 개봉일: 2025-06-09
   - 평점: 6.0
   - ![포스터](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)



In [6]:
# Test 2: 특정 영화 정보
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
user_input = "movie ID 550에 해당하는 영화가 무엇인지 알려줘"
print(f"User: {user_input}")
messages.append({"role": "user", "content": user_input})
call_ai(messages)

User: movie ID 550에 해당하는 영화가 무엇인지 알려줘
Calling function: get_movie_details with args: {"id":550}
AI: 영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 이 영화는 다음과 같은 정보가 있습니다:

- **개요**: 불면증에 시달리는 주인공과 미끄러운 비누 판매자가 원초적 남성 공격성을 새로운 형태의 치료로 채널링하는 이야기입니다. 이들의 개념은 각 도시에서 지하 "격투 클럽"이 형성될 정도로 인기를 끌게 되지만, 한 기이한 인물이 등장하면서 통제 불가능한 소용돌이에 휘말리게 됩니다.
- **장르**: 드라마, 스릴러
- **개봉일**: 1999년 10월 15일
- **러닝타임**: 139분
- **예산**: 63,000,000달러
- **수익**: 100,853,753달러
- **평점**: 8.4 (총 31,495표)
- **태그라인**: Mischief. Mayhem. Soap.
- **홈페이지**: [Fight Club](http://www.foxmovies.com/movies/fight-club)

![Fight Club](https://image.tmdb.org/t/p/w780/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg)


In [7]:
# Test 3: 특정 영화 출연진
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
user_input = "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
print(f"User: {user_input}")
messages.append({"role": "user", "content": user_input})
call_ai(messages)

User: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
Calling function: get_movie_credits with args: {"id":550}
AI: 영화 ID 550에 해당하는 영화는 **Fight Club**입니다. 주요 출연진은 다음과 같습니다:

1. **Edward Norton** - Narrator  
   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **Brad Pitt** - Tyler Durden  
   ![Brad Pitt](https://image.tmdb.org/t/p/w185/cckcYc2v0yh1tc9QjRelptcOBko.jpg)

3. **Helena Bonham Carter** - Marla Singer  
   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **Meat Loaf** - Robert Paulson  
   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **Jared Leto** - Angel Face  
   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

이 외에도 여러 배우들이 출연하고 있습니다. 추가적인 정보가 필요하시면 말씀해 주세요!
